# RAG: um assistente de controle de infecção hospitalar que consulta os documentos da instituição

**RAG** (*Retrieval-Augmented Generation*, geração aumentada por recuperação) é a técnica de **buscar trechos relevantes numa base de documentos e entregá-los ao modelo de linguagem junto com a pergunta**. O modelo não é treinado de novo: ele responde lendo os trechos e cita de onde tirou a informação.

Neste notebook montamos um RAG completo sobre uma pequena base de conhecimento de **epidemiologia hospitalar e infectologia** (indicadores, vigilância das IRAS, infecção de corrente sanguínea, resistência antimicrobiana, antibiograma, precauções, surtos, higiene das mãos e uso racional de antimicrobianos):

1. carregar os documentos;
2. dividir em trechos (*chunks*);
3. transformar cada trecho num vetor (*embedding*);
4. guardar os vetores num índice **FAISS**;
5. para cada pergunta, buscar os trechos mais parecidos;
6. pedir ao **DeepSeek** uma resposta baseada **só** nesses trechos, com as fontes;
7. comparar respostas **sem** e **com** RAG.

## 0. Como rodar este notebook

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasteurjr/notebookvideoaulas/blob/main/07_rag/notebook/rag_infeccao_hospitalar.ipynb)

O curso tem duas partes, usadas por **todos** os notebooks:

| O quê | Onde |
|---|---|
| **Código** (notebooks, arquivos de configuração) | GitHub: **https://github.com/pasteurjr/notebookvideoaulas** |
| **Dados** (imagens e arquivos grandes) | Google Drive: **[pasta notebooksvideos](https://drive.google.com/drive/folders/1mMRjinjohj_jyMsJzO7rxgYKm6unVKze?usp=sharing)**, com a mesma estrutura de pastas do GitHub |

**Esta aula usa:** nenhum dado externo (só o código do GitHub); não precisa de GPU; chaves de API: `DEEPSEEK_API_KEY`.

### Opção A: Google Colab (recomendado, nada para instalar no computador)

1. **Adicione a pasta de dados do curso ao seu Google Drive** (uma vez só, vale para todas as aulas): abra o link **[notebooksvideos](https://drive.google.com/drive/folders/1mMRjinjohj_jyMsJzO7rxgYKm6unVKze?usp=sharing)**, clique com o botão direito no nome da pasta e escolha **Organizar → Adicionar atalho** (ou **Adicionar atalho ao Drive**) → **Meu Drive**. A pasta passa a aparecer como `MyDrive/notebooksvideos`, sem ocupar o seu espaço.
2. **Abra este notebook no Colab** pelo botão **Abrir no Colab** acima (ele abre direto do GitHub).
3. **Guarde as chaves de API nos *Secrets* do Colab:** clique no ícone de **chave (🔑)** na barra lateral esquerda, crie `DEEPSEEK_API_KEY` com as **suas** chaves (com crédito) e ative **Acesso ao notebook** em cada uma. As chaves ficam na sua conta Google, nunca no notebook.
4. **Rode a célula de preparação logo abaixo.** Ela clona o repositório do GitHub para o Colab, verifica quais bibliotecas a sessão do Colab já tem e **instala só as que faltarem**, e carrega as chaves dos *Secrets*.
5. Depois: **Ambiente de execução → Executar tudo**. Rodar a célula de preparação de novo não causa problema.

**Se aparecer erro de biblioteca** (`ModuleNotFoundError`, `ImportError` ou erro de versão): crie uma célula nova, rode o comando abaixo, depois vá em **Ambiente de execução → Reiniciar sessão** e rode a célula de preparação de novo.

```
!pip install sentence-transformers faiss-cpu openai python-dotenv
```

### Opção B: no seu computador

Pré-requisito: [Miniconda](https://docs.conda.io/en/latest/miniconda.html) (ou Anaconda) e Git instalados. No terminal:

```bash
git clone https://github.com/pasteurjr/notebookvideoaulas.git
cd notebookvideoaulas
conda create -n notellm python=3.12 -y
conda activate notellm
pip install sentence-transformers faiss-cpu openai python-dotenv pandas jupyter
cd 07_rag/notebook
jupyter notebook rag_infeccao_hospitalar.ipynb
```

**Chaves:** copie `.env.exemplo` para `.env` e coloque nele as **suas** chaves (`DEEPSEEK_API_KEY`). O `.env` nunca vai para o GitHub.

No computador local, a célula de preparação detecta que não está no Colab e não faz nada.

In [1]:
# PREPARAÇÃO DO AMBIENTE NO GOOGLE COLAB (igual em todos os notebooks do curso; no computador local não faz nada)
AULA = "07_rag/notebook"                           # pasta desta aula no repositório
USA_DADOS_DO_DRIVE = False                                     # copia os dados da pasta "notebooksvideos" do Drive
PACOTES = {"sentence_transformers": "sentence-transformers", "faiss": "faiss-cpu", "openai": "openai", "dotenv": "python-dotenv"}   # módulo: pacote pip
SECRETS = ["DEEPSEEK_API_KEY"]                                   # chaves lidas dos Secrets do Colab (ícone 🔑)

import os, sys, shutil, subprocess, importlib.util

if "google.colab" in sys.modules:
    # 1) Código: clona o repositório do GitHub para o Colab
    if not os.path.exists("/content/notebookvideoaulas"):
        subprocess.run(["git", "clone", "-q", "https://github.com/pasteurjr/notebookvideoaulas.git",
                        "/content/notebookvideoaulas"], check=True)
    os.chdir(f"/content/notebookvideoaulas/{AULA}")

    # 2) Dados: copia da pasta "notebooksvideos" (atalho no Meu Drive) para a pasta da aula
    if USA_DADOS_DO_DRIVE and not os.path.exists("dados"):
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copytree(f"/content/drive/MyDrive/notebooksvideos/{AULA}/dados", "dados")

    # 3) Bibliotecas: instala só as que a sessão do Colab ainda não tem
    faltando = [pip for modulo, pip in PACOTES.items() if importlib.util.find_spec(modulo) is None]
    if faltando:
        print("Instalando:", ", ".join(faltando))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)
        print("Pronto. Se o Colab pedir 'Reiniciar sessão', reinicie e rode esta célula de novo.")
    else:
        print("Todas as bibliotecas já estão instaladas no Colab.")

    # 4) Chaves de API: vêm dos Secrets do Colab (nunca escreva a chave no notebook)
    if SECRETS:
        from google.colab import userdata
        for chave in SECRETS:
            os.environ[chave] = userdata.get(chave)

    print("Pasta da aula:", os.getcwd())
    print("Conteúdo:", sorted(os.listdir(".")))
else:
    print("Ambiente local detectado: usando a pasta atual.")

Ambiente local detectado: usando a pasta atual.


## 1. Bibliotecas e chave do DeepSeek

A chave fica no arquivo **`.env`** (no computador) ou nos **Secrets** do Colab, **nunca** no notebook. Crie uma chave **com crédito** em https://platform.deepseek.com/api_keys, copie `.env.exemplo` para `.env` e cole a chave em `DEEPSEEK_API_KEY`.

In [2]:
import os, re, textwrap
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # esconde as barras de download do modelo
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_VERBOSITY"] = "error"
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

load_dotenv()   # lê o .env da pasta da aula (no Colab a chave já veio dos Secrets)
llm = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")
MODELO_LLM = "deepseek-v4-flash"
pd.set_option("display.max_colwidth", 80)

def mostrar(texto, largura=110):
    print("\n".join(textwrap.fill(p, largura) for p in texto.split("\n")))

print("Chave do DeepSeek carregada:", bool(os.environ.get("DEEPSEEK_API_KEY")))

Chave do DeepSeek carregada: True


## 2. A base de conhecimento

São oito documentos curtos em Markdown, na pasta `dados/documentos`. Numa instituição real, seriam os protocolos da CCIH, os manuais e as normas técnicas.

In [3]:
PASTA = Path("dados/documentos")
documentos = {p.name: p.read_text(encoding="utf-8") for p in sorted(PASTA.glob("*.md"))}

pd.DataFrame([{"arquivo": nome, "título": texto.splitlines()[0].lstrip("# "), "caracteres": len(texto)}
              for nome, texto in documentos.items()])

,arquivo,título,caracteres
0,01_indicadores_epidemiologicos.md,Indicadores epidemiológicos básicos,2612
1,02_vigilancia_iras.md,Vigilância das infecções relacionadas à assistência à saúde (IRAS),2541
2,03_ipcs_cateter_central.md,Infecção primária de corrente sanguínea associada a cateter central,2445
3,04_resistencia_antimicrobiana.md,Resistência antimicrobiana e multirresistência,2377
4,05_antibiograma.md,O antibiograma e sua interpretação,2409
5,06_precaucoes_isolamento.md,Precauções e isolamento,2106
6,07_investigacao_de_surtos.md,Investigação de surtos,2313
7,08_higiene_maos_e_stewardship.md,Higiene das mãos e uso racional de antimicrobianos,2345


## 3. Dividir em trechos (*chunks*)

O modelo de embeddings e o próprio LLM funcionam melhor com trechos curtos, cada um sobre um assunto. Aqui cortamos cada documento **nas suas seções** (`## ...`); uma seção longa é dividida por parágrafos. Cada trecho começa com o título do documento e da seção, e guarda o nome do arquivo para a resposta poder citar a fonte.

In [4]:
TAMANHO_MAX = 1500   # caracteres por trecho

def dividir(nome, texto):
    titulo = texto.splitlines()[0].lstrip("# ")
    trechos = []
    for secao in re.split(r"\n(?=## )", texto):
        linhas = secao.strip().splitlines()
        nome_secao = linhas[0].lstrip("# ")
        cabecalho = titulo if nome_secao == titulo else f"{titulo} - {nome_secao}"
        atual = ""
        for paragrafo in "\n".join(linhas[1:]).strip().split("\n\n") + [None]:   # None fecha o último trecho
            if atual and (paragrafo is None or len(atual) + len(paragrafo) > TAMANHO_MAX):
                trechos.append({"fonte": nome, "secao": nome_secao, "texto": f"{cabecalho}\n{atual}"})
                atual = ""
            if paragrafo is not None:
                atual = (atual + "\n\n" + paragrafo).strip()
    return trechos

trechos = [t for nome, texto in documentos.items() for t in dividir(nome, texto)]
print(len(trechos), "trechos, de", min(len(t["texto"]) for t in trechos), "a", max(len(t["texto"]) for t in trechos), "caracteres\n")
mostrar(trechos[25]["texto"])

42 trechos, de 242 a 1405 caracteres

O antibiograma e sua interpretação - Como é feito
Os métodos mais usados são:
- **disco-difusão**: discos com antibiótico são colocados sobre a placa com a bactéria; mede-se o diâmetro do
halo de inibição ao redor de cada disco;
- **determinação da concentração inibitória mínima (CIM)**: a menor concentração do antibiótico que impede o
crescimento visível da bactéria, obtida por microdiluição, fitas de gradiente ou equipamentos automatizados.

Os valores medidos são comparados com **pontos de corte** (breakpoints) publicados por comitês técnicos. No
Brasil, adota-se o **BrCAST**, que segue as normas do comitê europeu (EUCAST).


## 4. Embeddings: cada trecho vira um vetor

Um **modelo de embeddings** transforma um texto num vetor de números, de forma que **textos com significado parecido ficam com vetores próximos**. Usamos um modelo multilíngue pequeno, que entende português e roda bem na CPU. Os vetores são normalizados, e assim o produto interno entre dois deles é a **similaridade do cosseno**.

In [5]:
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vetores = embedder.encode([t["texto"] for t in trechos], normalize_embeddings=True, show_progress_bar=False)
print("Matriz de embeddings:", vetores.shape, "(trechos x dimensões)")
print("Primeiros valores do trecho 0:", np.round(vetores[0][:6], 3))

Matriz de embeddings: (42, 384) (trechos x dimensões)
Primeiros valores do trecho 0: [ 0.018 -0.003 -0.007  0.035  0.158 -0.02 ]


## 5. O índice FAISS

O **FAISS** (biblioteca da Meta) guarda os vetores e encontra rapidamente os mais próximos de um vetor de consulta. Com poucos trechos, o índice exato `IndexFlatIP` (produto interno) basta; com milhões de trechos, o FAISS oferece índices aproximados, muito mais rápidos.

In [6]:
indice = faiss.IndexFlatIP(vetores.shape[1])
indice.add(vetores)
print("Vetores no índice:", indice.ntotal)

def buscar(pergunta, k=3):
    consulta = embedder.encode([pergunta], normalize_embeddings=True)
    notas, posicoes = indice.search(consulta, k)
    return [dict(trechos[i], similaridade=round(float(s), 3)) for s, i in zip(notas[0], posicoes[0])]

pergunta = "Quando devo usar respirador N95 em vez de máscara cirúrgica?"
pd.DataFrame(buscar(pergunta))[["similaridade", "fonte", "secao"]]

Vetores no índice: 42


,similaridade,fonte,secao
0,0.617,06_precaucoes_isolamento.md,Precauções para aerossóis
1,0.502,06_precaucoes_isolamento.md,Precauções para gotículas
2,0.469,06_precaucoes_isolamento.md,Precauções padrão


## 6. Geração: o DeepSeek responde lendo os trechos

O *prompt* leva os trechos encontrados e uma instrução clara: **responder só com base neles**, dizer quando a informação não está na base e **citar a fonte** de cada afirmação.

In [7]:
INSTRUCAO = ("Você é o assistente da Comissão de Controle de Infecção Hospitalar. "
             "Responda em português, em no máximo 5 frases, usando SOMENTE os trechos fornecidos. "
             "Cite a fonte de cada informação entre colchetes, por exemplo [05_antibiograma.md]. "
             "Se a resposta não estiver nos trechos, diga que não encontrou essa informação na base.")

def responder_com_rag(pergunta, k=3):
    encontrados = buscar(pergunta, k)
    contexto = "\n\n".join(f"[{t['fonte']}]\n{t['texto']}" for t in encontrados)
    r = llm.chat.completions.create(model=MODELO_LLM, temperature=0, messages=[
        {"role": "system", "content": INSTRUCAO},
        {"role": "user", "content": f"TRECHOS:\n{contexto}\n\nPERGUNTA: {pergunta}"}])
    return r.choices[0].message.content, encontrados

def responder_sem_rag(pergunta):
    r = llm.chat.completions.create(model=MODELO_LLM, temperature=0, messages=[
        {"role": "system", "content": "Responda em português, em no máximo 5 frases."},
        {"role": "user", "content": pergunta}])
    return r.choices[0].message.content

resposta, fontes = responder_com_rag(pergunta)
mostrar(resposta)

Use respirador **N95 ou PFF2** para o profissional nas precauções para aerossóis, indicadas para tuberculose
pulmonar, sarampo e varicela [06_precaucoes_isolamento.md]. Use **máscara cirúrgica** nas precauções para
gotículas, como influenza, doença meningocócica e coqueluche, ao se aproximar do paciente
[06_precaucoes_isolamento.md]. Nas precauções padrão, a máscara é indicada quando houver risco de respingos
[06_precaucoes_isolamento.md].


## 7. Sem RAG e com RAG

A vantagem do RAG aparece quando a pergunta depende **das regras da instituição**. No antibiograma abaixo, o nosso protocolo de vigilância conta **só o resultado R** como resistência; o resultado **I** não entra na contagem. Sem RAG, o modelo responde com o critério geral que aprendeu, e pode agrupar as classes de outro jeito; com RAG, ele aplica o protocolo da instituição e mostra de onde tirou.

In [8]:
caso = ("Um Staphylococcus aureus veio no antibiograma resistente (R) a uma cefalosporina, a um betalactâmico, "
        "a uma fluoroquinolona e a um aminoglicosídeo, e com resultado I para um carbapenêmico. "
        "Pelo protocolo de vigilância, quantas classes contam como resistentes? Ele é multirresistente?")

print("=== SEM RAG ===")
mostrar(responder_sem_rag(caso))
print("\n=== COM RAG ===")
resposta, fontes = responder_com_rag(caso)
mostrar(resposta)
print("\nTrechos usados:", [(t["fonte"], t["similaridade"]) for t in fontes])

=== SEM RAG ===


Contam como resistentes 3 classes: betalactâmicos (cefalosporina e carbapenêmico entram aqui),
fluoroquinolonas e aminoglicosídeos. O resultado I para o carbapenêmico não adiciona classe, pois ele também é
betalactâmico. Portanto, sim, ele é multirresistente (MDR), por ter resistência a pelo menos três classes.

=== COM RAG ===


Pelo protocolo de vigilância, apenas o resultado **R** conta como resistência; o resultado **I** não entra na
contagem [05_antibiograma.md]. Nesse caso, contam-se **quatro classes resistentes** (cefalosporina,
betalactâmico, fluoroquinolona e aminoglicosídeo), pois o carbapenêmico com resultado I não é contado
[05_antibiograma.md]. Portanto, ele é classificado como **multirresistente** [05_antibiograma.md].

Trechos usados: [('04_resistencia_antimicrobiana.md', 0.836), ('04_resistencia_antimicrobiana.md', 0.807), ('05_antibiograma.md', 0.793)]


Um cálculo de indicador, e uma pergunta **fora da base**. Um bom RAG não inventa: quando os trechos não trazem a resposta, ele diz isso.

In [9]:
for pergunta in ["Na UTI, em setembro, houve 3 infecções de corrente sanguínea associadas a cateter central e 750 cateter-dia. "
                 "Qual é a densidade de incidência?",
                 "Qual é a dose de vancomicina para um adulto de 70 kg?"]:
    resposta, fontes = responder_com_rag(pergunta)
    print("PERGUNTA:", pergunta)
    mostrar("RAG: " + resposta)
    print()

PERGUNTA: Na UTI, em setembro, houve 3 infecções de corrente sanguínea associadas a cateter central e 750 cateter-dia. Qual é a densidade de incidência?
RAG: A densidade de incidência é calculada dividindo o número de infecções pela soma de cateter-dia e
multiplicando por 1000 [01_indicadores_epidemiologicos.md]. Com 3 infecções e 750 cateter-dia, tem-se 3 ÷ 750
× 1000 = 4 infecções por 1000 cateter-dia. O indicador principal de IPCS associada a cateter central é
expresso por 1000 cateter-dia [03_ipcs_cateter_central.md].



PERGUNTA: Qual é a dose de vancomicina para um adulto de 70 kg?
RAG: Não encontrei essa informação na base. Os trechos disponíveis abordam apenas densidade de incidência e
taxa de utilização de dispositivo [01_indicadores_epidemiologicos.md] e as categorias de interpretação do
antibiograma (S, I, R) [05_antibiograma.md]. Nenhum deles traz esquemas posológicos ou doses de
antimicrobianos, como a vancomicina. Para essa orientação, consulte o protocolo institucional de
antimicrobianos ou a farmácia clínica.



## 8. Conclusão

- O **RAG** conecta um LLM genérico aos **documentos da instituição**, sem treinar o modelo de novo.
- O fluxo é sempre o mesmo: **dividir → embeddings → índice vetorial → busca → geração com as fontes**.
- Atualizar o conhecimento é simples: basta trocar os documentos e reconstruir o índice.
- As respostas citam as fontes e o modelo diz quando a informação não está na base: isso dá **rastreabilidade**, essencial em saúde.

**Experimente:** acrescente um documento seu em `dados/documentos` (por exemplo, o protocolo de antibioticoprofilaxia cirúrgica), rode tudo de novo e faça perguntas sobre ele.